# Blokus Duo AlphaZero — Colab training

Trains the policy/value network by self-play on a GPU, checkpointing to Google
Drive so progress survives Colab's session limits. **If the session drops, just
re-run the training cell — it resumes from the latest checkpoint.**

Set the runtime to **GPU**: Runtime → Change runtime type → T4 GPU.

## 1. Get the code

In [ ]:
# After you push this repo to GitHub, set REPO_URL and run this cell.
REPO_URL = "https://github.com/Kaikai064/blokus-duo-ai.git"  # <- your repo URL
import os
repo = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo

In [ ]:
# torch & numpy are preinstalled on Colab; numba usually is too (install to be safe).
!pip install -q numba

## 2. Mount Drive for checkpoints

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/blokus_ckpt'
import os
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoints ->', CKPT_DIR)

## 3. Validate the engine (incl. the Numba move-gen fuzz test)

Run the test suite before enabling the Numba fast path for a real run.

In [ ]:
!python -m pytest -q

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 4. Train

Self-play → train → checkpoint, each iteration. Logs loss and (every
`eval_every` iters) win rate vs the Random and Blocking baselines. Shrink
`channels`/`blocks`/`n_sims`/`games_per_iter` for a quick smoke run first.

In [ ]:
from blokus_core import board
from training.config import Config
from training import loop

board.set_numba(True)   # safe: validated by the tests above

cfg = Config(
    ckpt_dir=CKPT_DIR,
    channels=96, blocks=8,
    n_sims=128, games_per_iter=64,
    train_steps_per_iter=400, batch_size=256,
    num_iters=200, eval_every=5,
)
loop.run(cfg, resume=True)

**Resuming:** if the runtime disconnects, reconnect, re-run cells 1–3, then
re-run the training cell. It loads the latest checkpoint from Drive and
continues.

## 5. Quick strength check of the latest checkpoint

In [ ]:
import random
from blokus_core.net import BlokusNet, NetEvaluator
from blokus_core.mcts import MCTSPlayer
from training.replay import ReplayBuffer
from eval.arena import play_match
from eval.baselines import RandomPlayer, GreedyBlockingPlayer

net = BlokusNet(cfg.channels, cfg.blocks)
opt = torch.optim.AdamW(net.parameters())
loop.load_checkpoint(cfg, net, opt, ReplayBuffer(cfg.buffer_size), random.Random(0))
ev = NetEvaluator(net, cfg.device)

def make_net():
    return MCTSPlayer(ev, n_sims=200, temperature=0.0,
                      rng=random.Random(random.randrange(10**9)))

for name, opp in [('Random', RandomPlayer), ('Blocking', GreedyBlockingPlayer)]:
    res = play_match(make_net,
                     lambda: opp(random.Random(random.randrange(10**9))),
                     num_pairs=25)
    print(f"vs {name}: score {res['score_rate']*100:.1f}%  {res}")